In [1]:
import pandas as pd
import numpy as np
import sys
import os
from tqdm import tqdm

In [7]:
import warnings 
warnings.filterwarnings("ignore")

In [8]:
sys.path.append('../src/scripts_shibani/')

In [3]:
from utils import load_sessions, read_session
from events import generate_event_seq
from summary import stats, print_summary_stats
from main import generate_buffer

In [4]:
os.getcwd()

'/Users/treuilli/Documents/GitHub/GenAI_Interaction_Dynamics_Authorship/notebooks'

In [5]:
sessions = load_sessions()

Successfully downloaded 1447 writing sessions in CoAuthor!


In [6]:
#Initialize the lists in which data will be stored
file_name = []
text = []
sentence_metrics_list = []
api_metrics_list = []

err = []

In [9]:
#Compute metrics for each session
for sess in tqdm(sessions):
    events = read_session(sess, verbose=0)
    try:
        text_buffer = generate_buffer(events)
    except:
        err.append(str(sess.split('/')[-1]) + " is throwing an error!")
        continue
    file_name.append(sess.split('/')[-1])
    text.append(text_buffer[-1])
    event_seq_dict = generate_event_seq(buffer=text_buffer,
                                        events=events)
    sentence_metrics, api_metrics = stats(event_seq_dict)
    sentence_metrics_list.append(sentence_metrics)
    api_metrics_list.append(api_metrics)
    
for e in err:
    print(e)

  0%|          | 0/1447 [00:00<?, ?it/s]

100%|██████████| 1447/1447 [05:24<00:00,  4.46it/s]

312e3263a9f24f3184364949a42a6dfc.jsonl is throwing an error!


In [10]:
#Save results in a dataframe
df = pd.DataFrame()

df["file_name"] = file_name
df["text"] = text

for col in sentence_metrics_list[0]:
    df[str(col)] = [x[col] for x in sentence_metrics_list]
    
for col in api_metrics_list[0]:
    df[str(col)] = [x[col] for x in api_metrics_list]

In [11]:
#Remove duplicates
df = df.drop_duplicates(ignore_index=True)

In [12]:
df = df.rename(columns={"Total number of sentences": "nb_sentences_total", "Number of sentences of initial prompt":"nb_sentences_initial_prompt", "Number of sentences completely authored by the user": "nb_sentences_full_user", 
                        "Number of sentences completely authored by GPT-3":"nb_sentences_full_GPT", 
                        "Number of sentences authored by GPT-3 and user":"nb_sentences_GPT_user",
                        "Total number of GPT-3 calls made":"nb_gpt_calls", "Number of times GPT-3 suggestion is used":"nb_used_suggestions", 
                        "Number of times user rejected GPT-3 suggestion":"nb_rejected_suggestions", 
                        "Number of times GPT-3 suggestion is modified":"nb_modified_suggestions",
                        "Number of times GPT-3 suggestion is used as is":"nb_accepted_as_it_is_suggestions"}, inplace=False)

In [13]:
#Save data to csv 
df.to_csv('../data/metrics.csv', index=False)